# V34: predeclared related-prompt confirmation
Conditional on passing primary V34. Both fixed recipes resume external-only checkpoints. Related groups remain wholly in training or validation; no submission. Public pool and balancing: Nishant Dahal and contributors, CC BY-SA 4.0. Backbones: Qwen. LoRA: Hu et al.; SupCon: Khosla et al.; QLoRA: Dettmers et al. The original V34 protocol and attribution are embedded unchanged.

In [ ]:
import subprocess,sys,importlib.metadata
from packaging.version import Version
try: version=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError: version=None
if version is not None and Version(version)<Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)

import glob,hashlib
from pathlib import Path
wheels=glob.glob('/kaggle/input/**/bitsandbytes-0.50.0-py3-none-manylinux_2_24_x86_64.whl',recursive=True)
assert len(wheels)==1,wheels
assert hashlib.sha256(Path(wheels[0]).read_bytes()).hexdigest()=='173d137610468bec9cddbaa2e049254e97792657ab984e3e737bec1772c1668c'
subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall',wheels[0]],check=True)
assert importlib.metadata.version('bitsandbytes')=='0.50.0'


In [ ]:
import os, sys, subprocess, json, hashlib, time, signal
from pathlib import Path
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[key]='2'
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
Path('run_confirmation_arm_v34.py').write_text('"""Run one fixed recipe on independent host folds; each fit resumes external-only."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nfrom prepare_likelihood_v33 import prepare\n\n\ndef main():\n    root = Path.cwd()\n    plan = json.loads((root/\'plan.json\').read_text())\n    arm = sys.argv[1]\n    spec = plan[\'arms\'][arm]\n    records = []\n    for fold in plan[\'confirmation_folds\']:\n        folder = root/arm/f\'fold-{fold}\'\n        assert not folder.exists(), \'Never overwrite an existing fold run\'\n        folder.mkdir(parents=True)\n        params = dict(spec[\'parameters\'])\n        params.update(experiment=\'improvement-v34\', pilot_fold=fold,\n                      labels=plan[\'labels\'], model_dir=spec[\'model_dir\'],\n                      checkpoint=spec[\'checkpoint\'], checkpoint_sha256=spec[\'checkpoint_sha256\'],\n                      confirmation_split=f\'related-{fold}\', confirmation_fold=fold,\n                      host_sha256=plan[\'host_sha256\'], **plan[\'fold_sizes\'][str(fold)])\n        prepare(plan[\'host_path\'], plan[\'fold_by_prompt\'], params, folder)\n        (folder/\'training-plan.json\').write_text(json.dumps(params,indent=2))\n        (folder/\'plan.json\').write_text(json.dumps(params,indent=2))\n        worker = root/spec[\'training_worker\']\n        env = dict(os.environ,V18_VARIANT=spec[\'variant\'])\n        def run(path, environment):\n            remaining = plan[\'deadline_unix\']-time.time()\n            if remaining <= 0:\n                raise TimeoutError(\'Confirmation notebook wall budget exhausted\')\n            subprocess.run([sys.executable,\'-u\',str(path)],cwd=folder,env=environment,\n                           check=True,timeout=remaining)\n        run(worker,env)\n        training = json.loads((folder/spec[\'variant\']/\'result.json\').read_text())\n        assert training[\'completed\'] and training[\'validation_rows\']==params[\'validation_rows\']\n        readout = dict(params)\n        readout.update(spec[\'readout\'])\n        readout.update(host_path=plan[\'host_path\'],upstream_path=str(folder))\n        weight = folder/spec[\'variant\']/\'final/adapter/adapter_model.safetensors\'\n        readout[\'adapter_sha256\'] = {spec[\'readout_variant\']:hashlib.sha256(weight.read_bytes()).hexdigest()}\n        (folder/\'readout-plan.json\').write_text(json.dumps(readout,indent=2))\n        (folder/\'plan.json\').write_text(json.dumps(readout,indent=2))\n        try:\n            run(root/spec[\'readout_worker\'],dict(env,V19_VARIANTS=json.dumps([spec[\'readout_variant\']])))\n        finally:\n            (folder/\'plan.json\').write_text(json.dumps(params,indent=2))\n        result = json.loads((folder/(spec[\'readout_variant\']+\'_result.json\')).read_text())\n        record = {\'arm\':arm,\'fold\':fold,\'training\':training,\'readout\':result}\n        records.append(record)\n        with (root/(arm+\'-progress.jsonl\')).open(\'a\') as f:\n            f.write(json.dumps(record)+\'\\n\')\n        print(json.dumps(record),flush=True)\n    (root/(arm+\'-complete.json\')).write_text(json.dumps({\'completed\':True,\'folds\':records},indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n')
Path('prepare_likelihood_v33.py').write_text('"""Prepare one fixed host holdout, retaining related-prompt group boundaries."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\n\n\ndef prepare(host_path, split_by_prompt, plan, output):\n    host_path = Path(host_path)\n    assert hashlib.sha256(host_path.read_bytes()).hexdigest() == plan[\'host_sha256\']\n    host = pd.read_csv(host_path, dtype={\'id\': str, \'prompt_id\': str})\n    assert len(host) == 2699 and host.id.is_unique\n    assert set(host.prompt_id) == set(split_by_prompt)\n    labels = sorted(host.model.unique())\n    assert labels == plan[\'labels\'] and len(labels) == 18\n    host[\'group\'] = host.prompt_id.map(lambda p: str(split_by_prompt[p][\'group\']))\n    assigned = host.prompt_id.map(lambda p: int(split_by_prompt[p][\'fold\']))\n    assert host.assign(fold=assigned).groupby(\'group\').fold.nunique().max() == 1\n    heldout = assigned == plan[\'confirmation_fold\']\n    host[\'stage\'] = \'host\'\n    host[\'label\'] = host.model.map({name: i for i, name in enumerate(labels)}).astype(int)\n    train = host.loc[~heldout].reset_index(drop=True)\n    valid = host.loc[heldout].reset_index(drop=True)\n    assert len(train) == plan[\'training_rows\'] and len(valid) == plan[\'validation_rows\']\n    assert set(train.group).isdisjoint(valid.group)\n    assert set(train.prompt_id).isdisjoint(valid.prompt_id)\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    paths = [output / name for name in [\'training_frame.parquet\', \'validation_frame.parquet\']]\n    assert not any(p.exists() for p in paths), \'Existing prepared frames must not be overwritten\'\n    for path, frame in zip(paths, [train, valid]):\n        frame.to_parquet(path, index=False)\n    audit = {\'split\': plan[\'confirmation_split\'], \'training_rows\': len(train),\n             \'validation_rows\': len(valid), \'training_groups\': train.group.nunique(),\n             \'validation_groups\': valid.group.nunique(), \'labels\': labels,\n             \'sha256\': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}}\n    (output / \'data-audit.json\').write_text(json.dumps(audit, indent=2))\n    return audit\n')
Path('8b_ce_worker.py').write_text('"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef supcon_loss(features, labels, temperature=0.1):\n    """Mean log-probability of same-class positives, excluding self pairs."""\n    z = F.normalize(features.float(), dim=1)\n    logits = z @ z.T / temperature\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    counts = positives.sum(1)\n    if not bool((counts > 0).all()):\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\n\n\ndef paired_batches(frame, steps, seed):\n    """Two authors, two different prompts each; identical schedule for both arms."""\n    rng = np.random.default_rng(seed)\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\n              for label, group in frame.groupby("label")}\n    assert all(len(g) >= 2 for g in groups.values())\n    labels = sorted(groups)\n    for _ in range(steps):\n        selected = []\n        for label in rng.choice(labels, 2, replace=False):\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\n                selected.append(int(rng.choice(groups[label][prompt])))\n        yield selected\n\n\nclass AttributionModel(nn.Module):\n    def __init__(self, model_dir, plan):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n        from transformers import AutoModel, BitsAndBytesConfig\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\n            quantization_config=quant, device_map={"":0},\n            local_files_only=True, attn_implementation="sdpa")\n        assert backbone.config.hidden_size == 4096 and backbone.config.num_hidden_layers == 36\n        backbone = prepare_model_for_kbit_training(backbone,\n            gradient_checkpointing_kwargs={"use_reentrant":False})\n        backbone.config.use_cache = False\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\n            bias="none"))\n        width = backbone.config.hidden_size\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, 18)\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\n\n    def forward(self, batch):\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch["attention_mask"].clone()\n        mask[:, 0] = 0\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        features = self.norm(pooled)\n        return self.head(features), self.projection(features)\n\n\ndef main():\n    import pandas as pd\n    from transformers import AutoTokenizer\n    import transformers, peft\n    plan = json.loads(Path("plan.json").read_text())\n    variant = os.environ["V18_VARIANT"]\n    assert variant in plan["variants"]\n    out = Path(variant)\n    out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**kw):\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\n        print(json.dumps(record), flush=True)\n        with (out / "progress.jsonl").open("a") as f:\n            f.write(json.dumps(record)+"\\n")\n    assert torch.cuda.device_count() == 1\n    torch.manual_seed(plan["seed"])\n    np.random.seed(plan["seed"])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    model_dir = plan["model_dir"]\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    frame = pd.read_parquet("training_frame.parquet")\n    valid = pd.read_parquet("validation_frame.parquet")\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\n    def tokenize(df):\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                  add_special_tokens=False)\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\n    def batch(ids, bank):\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\n                                              padding=True, return_tensors="pt").items()}\n    model = AttributionModel(model_dir, plan)\n    model.norm.cuda(); model.head.cuda(); model.projection.cuda()\n    # Restore only an independently verified external-only checkpoint.\n    import hashlib\n    from safetensors.torch import load_file\n    from peft import set_peft_model_state_dict\n    checkpoint = Path(plan[\'checkpoint\'])\n    for relative,digest in plan[\'checkpoint_sha256\'].items():\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\n    state = load_file(str(checkpoint/\'adapter/adapter_model.safetensors\'))\n    restored = set_peft_model_state_dict(model.encoder,state)\n    assert not restored.unexpected_keys, restored.unexpected_keys\n    assert not [k for k in restored.missing_keys if \'lora_\' in k], restored.missing_keys\n    heads = torch.load(checkpoint/\'heads.pt\',map_location=\'cpu\',weights_only=True)\n    for name in [\'norm\',\'head\',\'projection\']:\n        getattr(model,name).load_state_dict(heads[name],strict=True)\n    assert set(frame.stage)=={\'host\'}\n    assert len(frame)==plan[\'training_rows\'] and len(valid)==plan[\'validation_rows\']\n    assert frame.id.is_unique and valid.id.is_unique and set(frame.id).isdisjoint(valid.id)\n    log(event=\'external_checkpoint_loaded\',sha256=plan[\'checkpoint_sha256\'],\n        fit_rows=len(frame),validation_rows=len(valid))\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\n    for p in trainable:\n        p.data = p.data.float()\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\n    scaler = torch.amp.GradScaler("cuda", init_scale=1024)\n    skipped = 0\n    for stage, epochs in [("host", plan["host_epochs"])]:\n        subset = frame.loc[frame.stage == stage]\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\n        optimizer = torch.optim.AdamW([\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\n        for epoch in range(epochs):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            losses = []\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\n                with torch.autocast("cuda", dtype=torch.float16):\n                    logits, features = model(batch(ids, tokens))\n                    ce = F.cross_entropy(logits.float(), labels)\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss")\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\n                scaler.scale(loss/divisor).backward()\n                losses.append(float(loss.detach()))\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n                    before = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    skipped += int(scaler.get_scale() < before)\n                    optimizer.zero_grad(set_to_none=True)\n                if step % 50 == 0:\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\n        model.encoder.save_pretrained(checkpoint / "adapter")\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\n    model.eval()\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(valid), 4):\n            ids = list(range(start, min(start+4, len(valid))))\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits, _ = model(batch(ids, val_tokens))\n            scores.append(logits.float().cpu().numpy())\n    scores = np.concatenate(scores)\n    assert np.isfinite(scores).all()\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\n                        targets=valid.label.to_numpy())\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\n              "validation_rows":len(valid), "completed":True}\n    (out / "result.json").write_text(json.dumps(result, indent=2))\n    log(event="complete", result=result)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('8b_ce_readout.py').write_text('"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel, prepare_model_for_kbit_training\n\n\ndef main():\n    plan = json.loads(Path("plan.json").read_text())\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\n    upstream = Path(plan["upstream_path"])\n    original_plan = json.loads((upstream / "plan.json").read_text())\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\n    val_ids = set(validation.id)\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\n    assert len(fit_ids) == plan["training_rows"] and len(val_ids) == plan["validation_rows"]\n    train_mask = host.id.isin(fit_ids).to_numpy()\n    valid_mask = host.id.isin(val_ids).to_numpy()\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\n    labels = np.asarray(original_plan["labels"])\n    assert np.array_equal(labels, np.sort(host.model.unique()))\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                    add_special_tokens=False)["input_ids"]\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\n        started = time.monotonic()\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\n        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\n            quantization_config=quant, device_map={"":0},\n            attn_implementation="sdpa", local_files_only=True)\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)\n        if variant != "base":\n            checkpoint = upstream / checkpoint_paths[variant]\n            weight = checkpoint / "adapter_model.safetensors"\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\n        model = model.eval()\n        width = model.config.hidden_size\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(host),plan["batch_size"]):\n                ids = order[start:start+plan["batch_size"]]\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\n                                                    padding=True,return_tensors="pt").items()}\n                with torch.autocast("cuda", dtype=torch.float16):\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\n                for j,layer in enumerate(plan["layers"]):\n                    h = states[layer].float()\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start % 400 == 0:\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\n        assert np.isfinite(features).all()\n        np.save(variant+"_features.npy", features)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n        predictions = []\n        diagnostics = {}\n        for j,layer in enumerate(plan["layers"]):\n            scaler = StandardScaler()\n            X = scaler.fit_transform(features[train_mask,j])\n            V = scaler.transform(features[valid_mask,j])\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                clf.fit(X,host.loc[train_mask,"model"])\n            assert np.array_equal(clf.classes_, labels)\n            scores = clf.predict_log_proba(V)\n            predictions.append(scores)\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        scores = np.mean(predictions,axis=0)\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n        del features\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n')
Path('4b_small_supcon_worker.py').write_text('"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef supcon_loss(features, labels, temperature=0.1):\n    """Mean log-probability of same-class positives, excluding self pairs."""\n    z = F.normalize(features.float(), dim=1)\n    logits = z @ z.T / temperature\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    counts = positives.sum(1)\n    if not bool((counts > 0).all()):\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\n\n\ndef paired_batches(frame, steps, seed):\n    """Two authors, two different prompts each; identical schedule for both arms."""\n    rng = np.random.default_rng(seed)\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\n              for label, group in frame.groupby("label")}\n    assert all(len(g) >= 2 for g in groups.values())\n    labels = sorted(groups)\n    for _ in range(steps):\n        selected = []\n        for label in rng.choice(labels, 2, replace=False):\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\n                selected.append(int(rng.choice(groups[label][prompt])))\n        yield selected\n\n\nclass AttributionModel(nn.Module):\n    def __init__(self, model_dir, plan):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model\n        from transformers import AutoModel\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\n                                            local_files_only=True, attn_implementation="sdpa")\n        backbone.config.use_cache = False\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\n            bias="none"))\n        width = backbone.config.hidden_size\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, 18)\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\n\n    def forward(self, batch):\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch["attention_mask"].clone()\n        mask[:, 0] = 0\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        features = self.norm(pooled)\n        return self.head(features), self.projection(features)\n\n\ndef main():\n    import pandas as pd\n    from transformers import AutoTokenizer\n    import transformers, peft\n    plan = json.loads(Path("plan.json").read_text())\n    variant = os.environ["V18_VARIANT"]\n    assert variant in plan["variants"]\n    out = Path(variant)\n    out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**kw):\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\n        print(json.dumps(record), flush=True)\n        with (out / "progress.jsonl").open("a") as f:\n            f.write(json.dumps(record)+"\\n")\n    assert torch.cuda.device_count() == 1\n    torch.manual_seed(plan["seed"])\n    np.random.seed(plan["seed"])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    model_dir = plan["model_dir"]\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    frame = pd.read_parquet("training_frame.parquet")\n    valid = pd.read_parquet("validation_frame.parquet")\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\n    def tokenize(df):\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                  add_special_tokens=False)\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\n    def batch(ids, bank):\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\n                                              padding=True, return_tensors="pt").items()}\n    model = AttributionModel(model_dir, plan).cuda()\n    # Restore only an independently verified external-only checkpoint.\n    import hashlib\n    from safetensors.torch import load_file\n    from peft import set_peft_model_state_dict\n    checkpoint = Path(plan[\'checkpoint\'])\n    for relative,digest in plan[\'checkpoint_sha256\'].items():\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\n    state = load_file(str(checkpoint/\'adapter/adapter_model.safetensors\'))\n    restored = set_peft_model_state_dict(model.encoder,state)\n    assert not restored.unexpected_keys, restored.unexpected_keys\n    assert not [k for k in restored.missing_keys if \'lora_\' in k], restored.missing_keys\n    heads = torch.load(checkpoint/\'heads.pt\',map_location=\'cpu\',weights_only=True)\n    for name in [\'norm\',\'head\',\'projection\']:\n        getattr(model,name).load_state_dict(heads[name],strict=True)\n    assert set(frame.stage)=={\'host\'}\n    assert len(frame)==plan[\'training_rows\'] and len(valid)==plan[\'validation_rows\']\n    assert frame.id.is_unique and valid.id.is_unique and set(frame.id).isdisjoint(valid.id)\n    log(event=\'external_checkpoint_loaded\',sha256=plan[\'checkpoint_sha256\'],\n        fit_rows=len(frame),validation_rows=len(valid))\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\n    for p in trainable:\n        p.data = p.data.float()\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\n    scaler = torch.amp.GradScaler("cuda")\n    skipped = 0\n    for stage, epochs in [("host", plan["host_epochs"])]:\n        subset = frame.loc[frame.stage == stage]\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\n        optimizer = torch.optim.AdamW([\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\n        for epoch in range(epochs):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            losses = []\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\n                with torch.autocast("cuda", dtype=torch.float16):\n                    logits, features = model(batch(ids, tokens))\n                    ce = F.cross_entropy(logits.float(), labels)\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss")\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\n                scaler.scale(loss/divisor).backward()\n                losses.append(float(loss.detach()))\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n                    before = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    skipped += int(scaler.get_scale() < before)\n                    optimizer.zero_grad(set_to_none=True)\n                if step % 50 == 0:\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\n        model.encoder.save_pretrained(checkpoint / "adapter")\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\n    model.eval()\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(valid), 4):\n            ids = list(range(start, min(start+4, len(valid))))\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits, _ = model(batch(ids, val_tokens))\n            scores.append(logits.float().cpu().numpy())\n    scores = np.concatenate(scores)\n    assert np.isfinite(scores).all()\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\n                        targets=valid.label.to_numpy())\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\n              "validation_rows":len(valid), "completed":True}\n    (out / "result.json").write_text(json.dumps(result, indent=2))\n    log(event="complete", result=result)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('4b_small_supcon_readout.py').write_text('"""Matched frozen readouts for base and v18 adapters; one fixed host fold."""\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom transformers import AutoModel, AutoTokenizer\nfrom peft import PeftModel\n\n\ndef main():\n    plan = json.loads(Path("plan.json").read_text())\n    host = pd.read_csv(plan["host_path"], dtype={"id":str})\n    upstream = Path(plan["upstream_path"])\n    original_plan = json.loads((upstream / "plan.json").read_text())\n    assert original_plan["pilot_fold"] == plan["pilot_fold"]\n    fitting = pd.read_parquet(upstream / "training_frame.parquet")\n    validation = pd.read_parquet(upstream / "validation_frame.parquet")\n    fit_ids = set(fitting.loc[fitting.stage == "host", "id"])\n    val_ids = set(validation.id)\n    assert fit_ids.isdisjoint(val_ids) and fit_ids | val_ids == set(host.id)\n    assert len(fit_ids) == plan["training_rows"] and len(val_ids) == plan["validation_rows"]\n    train_mask = host.id.isin(fit_ids).to_numpy()\n    valid_mask = host.id.isin(val_ids).to_numpy()\n    assert host.loc[valid_mask,"id"].tolist() == validation.id.tolist()\n    labels = np.asarray(original_plan["labels"])\n    assert np.array_equal(labels, np.sort(host.model.unique()))\n    tok = AutoTokenizer.from_pretrained(plan["model_dir"], local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None: tok.pad_token = tok.eos_token\n    tokenized = tok(host.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                    add_special_tokens=False)["input_ids"]\n    tokenized = [ids+[tok.eos_token_id] for ids in tokenized]\n    order = np.argsort([len(ids) for ids in tokenized], kind="stable")\n    checkpoint_paths = {"ce_external":"ce/external_only/adapter", "ce_final":"ce/final/adapter",\n                        "supcon_external":"ce_supcon/external_only/adapter", "supcon_final":"ce_supcon/final/adapter"}\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    for variant in json.loads(os.environ["V19_VARIANTS"]):\n        started = time.monotonic()\n        print(json.dumps({"variant":variant,"event":"loading"}), flush=True)\n        model = AutoModel.from_pretrained(plan["model_dir"], torch_dtype=torch.float16,\n                                         attn_implementation="sdpa", local_files_only=True)\n        if variant != "base":\n            checkpoint = upstream / checkpoint_paths[variant]\n            weight = checkpoint / "adapter_model.safetensors"\n            assert hashlib.sha256(weight.read_bytes()).hexdigest() == plan["adapter_sha256"][variant]\n            model = PeftModel.from_pretrained(model, checkpoint, is_trainable=False)\n        model = model.cuda().eval()\n        width = model.config.hidden_size\n        features = np.empty((len(host),len(plan["layers"]),width), dtype=np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(host),plan["batch_size"]):\n                ids = order[start:start+plan["batch_size"]]\n                enc = {k:v.cuda() for k,v in tok.pad({"input_ids":[tokenized[i] for i in ids]},\n                                                    padding=True,return_tensors="pt").items()}\n                with torch.autocast("cuda", dtype=torch.float16):\n                    states = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n                mask = enc["attention_mask"].clone(); mask[:,0] = 0\n                for j,layer in enumerate(plan["layers"]):\n                    h = states[layer].float()\n                    features[ids,j] = ((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start % 400 == 0:\n                    print(json.dumps({"variant":variant,"completed":min(start+len(ids),len(host)),\n                                      "total":len(host),"seconds":time.monotonic()-started}), flush=True)\n        assert np.isfinite(features).all()\n        np.save(variant+"_features.npy", features)\n        del model\n        gc.collect(); torch.cuda.empty_cache()\n        predictions = []\n        diagnostics = {}\n        for j,layer in enumerate(plan["layers"]):\n            scaler = StandardScaler()\n            X = scaler.fit_transform(features[train_mask,j])\n            V = scaler.transform(features[valid_mask,j])\n            clf = LogisticRegression(C=plan["C"], max_iter=plan["max_iter"], random_state=plan["seed"])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                clf.fit(X,host.loc[train_mask,"model"])\n            assert np.array_equal(clf.classes_, labels)\n            scores = clf.predict_log_proba(V)\n            predictions.append(scores)\n            diagnostics[str(layer)] = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        scores = np.mean(predictions,axis=0)\n        accuracy = float((labels[scores.argmax(1)]==host.loc[valid_mask,"model"]).mean())\n        np.savez_compressed(variant+"_validation.npz", ids=host.loc[valid_mask,"id"].to_numpy(dtype=str),\n                            labels=labels.astype(str), scores=scores, per_layer_scores=np.stack(predictions))\n        result = {"variant":variant,"accuracy":accuracy,"per_layer_accuracy":diagnostics,\n                  "seconds":time.monotonic()-started,"fit_rows":int(train_mask.sum()),\n                  "validation_rows":int(valid_mask.sum()),"feature_shape":list(features.shape),\n                  "feature_sha256":hashlib.sha256(Path(variant+"_features.npy").read_bytes()).hexdigest()}\n        Path(variant+"_result.json").write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n        del features\n        gc.collect()\n\n\nif __name__ == "__main__":\n    main()\n')
plan=json.loads("{\"experiment\": \"improvement-v34\", \"protocol\": {\"experiment\": \"improvement-v34\", \"kind\": \"Multi-fold comparison with the current incumbent component and fixed class balancing\", \"status\": \"planned_not_launched\", \"motivation\": \"Saved-prediction diagnostic exposed a difference between pilot and submitted postprocessing. Test two fixed training recipes across more prompts; do not retroactively pass V25.\", \"candidate\": {\"name\": \"8b_ce\", \"initialization\": \"V25 CE external-only checkpoint, including adapter and heads\", \"training_recipe\": \"Unchanged V25 CE host stage: two equivalent epochs; fresh optimizer and scaler; same paired sampler and seed\", \"readout\": \"Unchanged V25 layers 12/24/36 logistic readout, C=0.02, mean log probabilities\", \"pilot_scores\": \"artifacts/improvement-v25/kaggle-output/ce_final_validation.npz\"}, \"comparison\": {\"name\": \"4b_small_supcon\", \"initialization\": \"V18 CE+SupCon external-only checkpoint from 4500 selected external rows, including adapter and heads\", \"training_recipe\": \"Unchanged V18 CE+SupCon host stage: two equivalent epochs; fresh optimizer and scaler; same paired sampler and seed\", \"readout\": \"Unchanged V19 layers 12/24/36 logistic readout, C=0.02, mean log probabilities\", \"pilot_scores\": \"artifacts/improvement-v19/kaggle-output/supcon_final_validation.npz\", \"reason\": \"This recipe supplies V21's added component, our current 0.622 public incumbent; V22's larger-data component regressed publicly. This is still not validation of the full incumbent ensemble.\"}, \"choice_disclosure\": \"Candidate CE was chosen after observing V25: it ties SupCon on the balanced pilot blend and has the higher standalone readout. This is post-pilot selection, not a pre-pilot discovery claim.\", \"primary_folds\": {\"file\": \"artifacts/baseline-v1/folds.csv\", \"sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"reuse\": [0], \"train\": [1, 2, 3, 4]}, \"related_confirmation\": {\"file\": \"artifacts/improvement-v4/robustness/folds.csv\", \"sha256\": \"090c42e940fcfd380d821e2e7a6414605e301f826e5e6e29ad678be5643eee32\", \"fold\": 0, \"launch_requires\": \"All primary gates pass\"}, \"scoring\": {\"sparse_reference\": \"V32 fixed V17 sparse OOF; related holdout fits the identical specification using fixed V17 external IDs and train-only vocabulary\", \"component_weight\": 0.15, \"blend\": \"sparse decision scores + 0.15 * mean neural log probabilities\", \"postprocessing\": \"Softmax then credited V16 equal-class-mass balancing for exactly 300 iterations, separately on each held-out fold; no prompt-wise one-to-one assignment\", \"metric\": \"Balanced accuracy (mean recall across 18 source labels)\", \"no_selection\": \"No weight, epoch, layer, temperature, seed or objective search. Evaluate fixed final checkpoints. Report ordinary accuracy as a diagnostic only.\"}, \"primary_gate\": {\"candidate_pooled_balanced_accuracy_minimum\": 0.52, \"candidate_minus_comparison_minimum\": 0.02, \"materially_positive_gain_folds_minimum\": 4, \"material_tolerance\": 1e-08, \"paired_prompt_bootstrap_gain_lower_must_be_positive\": true}, \"related_gate\": {\"candidate_balanced_accuracy_minimum\": 0.5, \"candidate_minus_comparison_minimum\": 0.015}, \"bootstrap\": {\"samples\": 10000, \"seed\": 3401, \"unit\": \"Whole prompt groups, stratified by primary fold; recompute pooled balanced accuracy for each paired resample\", \"interval\": [0.025, 0.975]}, \"runtime\": {\"evidence\": \"V25 CE host stage through completion about 3811 seconds plus 557-second readout; V18 SupCon about 1483 seconds plus 264-second V19 readout\", \"execution\": \"One GPU per recipe, four primary folds sequentially within each recipe; related fold only after primary qualification\", \"primary_projection_seconds\": 18392, \"wall_limit_seconds\": 24300, \"launch_requires\": \"Independent verification of external-only artifacts, exact train/holdout boundaries, notebook source, offline model dependencies and sufficient current Kaggle quota\", \"paid_compute\": false}, \"attribution\": \"Public pool and fixed balancing: Nishant Dahal and original contributors, CC BY-SA 4.0. Qwen backbones: Qwen. LoRA, supervised contrastive learning and QLoRA retain the cited V18/V25 method credits. This is a new evaluation protocol, not a newly invented training method.\", \"limitations\": [\"Full credited 14B/DeBERTa ensemble is not included in local OOF; only an actual submission can prove a gain over 0.622\", \"Pilot fold zero is reused; all development labels have been inspected before\", \"Size, quantization, adapter coverage, objective and external-pool size differ between recipes; no pure scaling or causal method claim\", \"Equal hidden class mass is an assumption; balancing is fixed because the submitted baseline already uses it\", \"Related confirmation is one predeclared holdout, not full related-fold OOF\", \"Passing supports separate full-host training and inference validation; a fold model is never submitted directly\", \"V25 and V30 remain failed under their original gates; V33 remains unlaunched\"]}, \"protocol_sha256\": \"c5fde950434c1dd604a3233d9b989790d9af105de9c3d732172c73f3539a35db\", \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"], \"fold_by_prompt\": {\"p000\": {\"group\": \"p000\", \"fold\": 3}, \"p001\": {\"group\": \"p001\", \"fold\": 3}, \"p002\": {\"group\": \"p002\", \"fold\": 4}, \"p003\": {\"group\": \"p003\", \"fold\": 3}, \"p004\": {\"group\": \"p004\", \"fold\": 2}, \"p005\": {\"group\": \"p005\", \"fold\": 1}, \"p006\": {\"group\": \"p006\", \"fold\": 4}, \"p007\": {\"group\": \"p007\", \"fold\": 4}, \"p008\": {\"group\": \"p008\", \"fold\": 3}, \"p009\": {\"group\": \"p009\", \"fold\": 0}, \"p010\": {\"group\": \"p010\", \"fold\": 4}, \"p011\": {\"group\": \"p011\", \"fold\": 2}, \"p012\": {\"group\": \"p012\", \"fold\": 3}, \"p013\": {\"group\": \"p013\", \"fold\": 4}, \"p014\": {\"group\": \"p014\", \"fold\": 4}, \"p015\": {\"group\": \"p015\", \"fold\": 3}, \"p016\": {\"group\": \"p016\", \"fold\": 3}, \"p017\": {\"group\": \"p017\", \"fold\": 4}, \"p018\": {\"group\": \"p018\", \"fold\": 2}, \"p019\": {\"group\": \"p019\", \"fold\": 0}, \"p020\": {\"group\": \"p020\", \"fold\": 3}, \"p021\": {\"group\": \"p021\", \"fold\": 2}, \"p022\": {\"group\": \"p022\", \"fold\": 0}, \"p023\": {\"group\": \"p023\", \"fold\": 2}, \"p024\": {\"group\": \"p024\", \"fold\": 2}, \"p025\": {\"group\": \"p025\", \"fold\": 3}, \"p026\": {\"group\": \"p026\", \"fold\": 3}, \"p027\": {\"group\": \"p027\", \"fold\": 0}, \"p028\": {\"group\": \"p028\", \"fold\": 2}, \"p029\": {\"group\": \"p029\", \"fold\": 2}, \"p030\": {\"group\": \"p030\", \"fold\": 4}, \"p031\": {\"group\": \"p031\", \"fold\": 2}, \"p032\": {\"group\": \"p032\", \"fold\": 0}, \"p033\": {\"group\": \"p033\", \"fold\": 1}, \"p034\": {\"group\": \"p034\", \"fold\": 4}, \"p035\": {\"group\": \"p035\", \"fold\": 2}, \"p036\": {\"group\": \"p036\", \"fold\": 0}, \"p037\": {\"group\": \"p037\", \"fold\": 1}, \"p038\": {\"group\": \"p038\", \"fold\": 3}, \"p039\": {\"group\": \"p039\", \"fold\": 2}, \"p040\": {\"group\": \"p040\", \"fold\": 1}, \"p041\": {\"group\": \"p041\", \"fold\": 1}, \"p042\": {\"group\": \"p042\", \"fold\": 0}, \"p043\": {\"group\": \"p043\", \"fold\": 2}, \"p044\": {\"group\": \"p044\", \"fold\": 2}, \"p045\": {\"group\": \"p045\", \"fold\": 2}, \"p046\": {\"group\": \"p046\", \"fold\": 3}, \"p047\": {\"group\": \"p047\", \"fold\": 1}, \"p048\": {\"group\": \"p048\", \"fold\": 2}, \"p049\": {\"group\": \"p049\", \"fold\": 1}, \"p050\": {\"group\": \"p050\", \"fold\": 4}, \"p051\": {\"group\": \"p051\", \"fold\": 4}, \"p052\": {\"group\": \"p052\", \"fold\": 1}, \"p053\": {\"group\": \"p053\", \"fold\": 2}, \"p054\": {\"group\": \"p054\", \"fold\": 1}, \"p055\": {\"group\": \"p055\", \"fold\": 3}, \"p056\": {\"group\": \"p056\", \"fold\": 1}, \"p057\": {\"group\": \"p057\", \"fold\": 0}, \"p058\": {\"group\": \"p055\", \"fold\": 3}, \"p059\": {\"group\": \"p059\", \"fold\": 4}, \"p060\": {\"group\": \"p060\", \"fold\": 3}, \"p061\": {\"group\": \"p061\", \"fold\": 4}, \"p062\": {\"group\": \"p062\", \"fold\": 0}, \"p063\": {\"group\": \"p063\", \"fold\": 4}, \"p064\": {\"group\": \"p064\", \"fold\": 1}, \"p065\": {\"group\": \"p065\", \"fold\": 0}, \"p066\": {\"group\": \"p066\", \"fold\": 1}, \"p067\": {\"group\": \"p067\", \"fold\": 3}, \"p068\": {\"group\": \"p068\", \"fold\": 0}, \"p069\": {\"group\": \"p069\", \"fold\": 4}, \"p070\": {\"group\": \"p070\", \"fold\": 0}, \"p071\": {\"group\": \"p071\", \"fold\": 3}, \"p072\": {\"group\": \"p072\", \"fold\": 1}, \"p073\": {\"group\": \"p073\", \"fold\": 1}, \"p074\": {\"group\": \"p074\", \"fold\": 3}, \"p075\": {\"group\": \"p075\", \"fold\": 1}, \"p076\": {\"group\": \"p076\", \"fold\": 2}, \"p077\": {\"group\": \"p077\", \"fold\": 4}, \"p078\": {\"group\": \"p078\", \"fold\": 0}, \"p079\": {\"group\": \"p079\", \"fold\": 0}, \"p080\": {\"group\": \"p080\", \"fold\": 1}, \"p081\": {\"group\": \"p081\", \"fold\": 2}, \"p082\": {\"group\": \"p010\", \"fold\": 4}, \"p083\": {\"group\": \"p083\", \"fold\": 1}, \"p084\": {\"group\": \"p084\", \"fold\": 2}, \"p085\": {\"group\": \"p085\", \"fold\": 0}, \"p086\": {\"group\": \"p086\", \"fold\": 0}, \"p087\": {\"group\": \"p087\", \"fold\": 4}, \"p088\": {\"group\": \"p088\", \"fold\": 1}, \"p089\": {\"group\": \"p089\", \"fold\": 0}, \"p090\": {\"group\": \"p090\", \"fold\": 1}, \"p091\": {\"group\": \"p091\", \"fold\": 4}, \"p092\": {\"group\": \"p092\", \"fold\": 1}, \"p093\": {\"group\": \"p093\", \"fold\": 1}, \"p094\": {\"group\": \"p094\", \"fold\": 1}, \"p095\": {\"group\": \"p095\", \"fold\": 3}, \"p096\": {\"group\": \"p096\", \"fold\": 2}, \"p097\": {\"group\": \"p097\", \"fold\": 3}, \"p098\": {\"group\": \"p098\", \"fold\": 2}, \"p099\": {\"group\": \"p099\", \"fold\": 4}, \"p100\": {\"group\": \"p100\", \"fold\": 0}, \"p101\": {\"group\": \"p101\", \"fold\": 0}, \"p102\": {\"group\": \"p102\", \"fold\": 2}, \"p103\": {\"group\": \"p103\", \"fold\": 4}, \"p104\": {\"group\": \"p104\", \"fold\": 4}, \"p105\": {\"group\": \"p105\", \"fold\": 3}, \"p106\": {\"group\": \"p106\", \"fold\": 3}, \"p107\": {\"group\": \"p107\", \"fold\": 3}, \"p108\": {\"group\": \"p108\", \"fold\": 0}, \"p109\": {\"group\": \"p109\", \"fold\": 3}, \"p110\": {\"group\": \"p110\", \"fold\": 0}, \"p111\": {\"group\": \"p004\", \"fold\": 2}, \"p112\": {\"group\": \"p112\", \"fold\": 1}, \"p113\": {\"group\": \"p113\", \"fold\": 4}, \"p114\": {\"group\": \"p114\", \"fold\": 1}, \"p115\": {\"group\": \"p115\", \"fold\": 4}, \"p116\": {\"group\": \"p116\", \"fold\": 2}, \"p117\": {\"group\": \"p117\", \"fold\": 2}, \"p118\": {\"group\": \"p118\", \"fold\": 3}, \"p119\": {\"group\": \"p119\", \"fold\": 0}, \"p120\": {\"group\": \"p120\", \"fold\": 4}, \"p121\": {\"group\": \"p121\", \"fold\": 4}, \"p122\": {\"group\": \"p122\", \"fold\": 0}, \"p123\": {\"group\": \"p123\", \"fold\": 2}, \"p124\": {\"group\": \"p124\", \"fold\": 1}, \"p125\": {\"group\": \"p125\", \"fold\": 1}, \"p126\": {\"group\": \"p126\", \"fold\": 3}, \"p127\": {\"group\": \"p127\", \"fold\": 3}, \"p128\": {\"group\": \"p128\", \"fold\": 3}, \"p129\": {\"group\": \"p129\", \"fold\": 0}, \"p130\": {\"group\": \"p130\", \"fold\": 0}, \"p131\": {\"group\": \"p008\", \"fold\": 3}, \"p132\": {\"group\": \"p132\", \"fold\": 1}, \"p133\": {\"group\": \"p133\", \"fold\": 2}, \"p134\": {\"group\": \"p134\", \"fold\": 0}, \"p135\": {\"group\": \"p135\", \"fold\": 2}, \"p136\": {\"group\": \"p128\", \"fold\": 3}, \"p137\": {\"group\": \"p137\", \"fold\": 1}, \"p138\": {\"group\": \"p138\", \"fold\": 4}, \"p139\": {\"group\": \"p139\", \"fold\": 4}, \"p140\": {\"group\": \"p140\", \"fold\": 3}, \"p141\": {\"group\": \"p141\", \"fold\": 0}, \"p142\": {\"group\": \"p142\", \"fold\": 2}, \"p143\": {\"group\": \"p143\", \"fold\": 0}, \"p144\": {\"group\": \"p144\", \"fold\": 4}, \"p145\": {\"group\": \"p145\", \"fold\": 0}, \"p146\": {\"group\": \"p100\", \"fold\": 0}, \"p147\": {\"group\": \"p147\", \"fold\": 1}, \"p148\": {\"group\": \"p017\", \"fold\": 4}, \"p149\": {\"group\": \"p149\", \"fold\": 2}}, \"fold_sizes\": {\"0\": {\"training_rows\": 2159, \"validation_rows\": 540}}, \"arms\": {\"8b_ce\": {\"parameters\": {\"experiment\": \"improvement-v25\", \"kind\": \"8B NF4 all-linear paired CE/SupCon pilot\", \"model\": \"qwen-lm/qwen-3/Transformers/8b-base/1\", \"seed\": 1801, \"pilot_fold\": 0, \"external_cap_per_class\": 1000, \"external_host_cosine_exclusion\": 0.3, \"max_length\": 256, \"batch_size\": 4, \"gradient_accumulation\": 8, \"lora_rank\": 16, \"lora_alpha\": 32, \"target_modules\": \"all-linear\", \"pool_epochs\": 1, \"host_epochs\": 2, \"pool_adapter_lr\": 0.0001, \"pool_head_lr\": 0.0005, \"host_adapter_lr\": 3e-05, \"host_head_lr\": 0.0001, \"contrastive_weight\": 0.1, \"contrastive_temperature\": 0.1, \"variants\": [\"ce\", \"ce_supcon\"], \"checkpoint_rule\": \"Evaluate fixed final host epoch only; no best-epoch selection\", \"promotion\": \"Fixed three-layer readout must reach >=0.4625925926 and fixed 0.15 sparse-pilot blend >=0.5311111111. Select highest blend among qualified arms, report two-arm selection. No direct pilot submission.\", \"wall_limit_seconds\": 24300, \"sources\": [\"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast\", \"https://arxiv.org/html/2106.09685v2\", \"https://github.com/microsoft/LoRA\", \"https://huggingface.co/docs/peft/package_reference/lora\", \"https://arxiv.org/abs/2305.14314\", \"https://github.com/artidoro/qlora\", \"https://huggingface.co/docs/peft/developer_guides/quantization\", \"https://huggingface.co/Qwen/Qwen3-8B-Base/raw/main/config.json\"], \"attribution\": \"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0). QLoRA: Dettmers et al. 2305.14314; implementation uses Hugging Face PEFT/bitsandbytes. Uses ordinary AdamW and fp16 compute on T4.\", \"limitations\": [\"Same repeatedly inspected 540-row fold; not an untouched test\", \"Changes model size, quantization and adapter coverage together; no pure scaling attribution\", \"Two objectives compared; no full-ensemble CV claim\", \"QLoRA instruction-tuning results do not predict attribution accuracy\", \"No paid compute\"], \"comparison\": {\"v22_supcon_readout\": 0.4425925926, \"v22_supcon_sparse_blend\": 0.5111111111}, \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8}, \"differences_from_v18\": [\"18000 external rows as V22\", \"8B NF4 base with all-linear rank16 adapters instead of 4B fp16 q/v-only\", \"Unchanged fixed readout layers12/24/36 and C0.02\"], \"reason\": \"Test a stronger trainable reader, with matched CE and SupCon objectives; require two-point gains over V22 selected component and sparse blend.\", \"launch_gate\": \"V24 completed finite 8B NF4 updates and adapter reload; projected 5580 training microsteps plus conservative 1800s readout/loading allowance <=24300s (25% below 9h).\", \"quantization\": {\"load_in_4bit\": true, \"quant_type\": \"nf4\", \"double_quant\": true, \"compute_dtype\": \"float16\"}, \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"pool_sha256\": \"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\", \"folds_sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"]}, \"variant\": \"ce\", \"readout_variant\": \"ce_final\", \"training_worker\": \"8b_ce_worker.py\", \"readout_worker\": \"8b_ce_readout.py\", \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8}, \"checkpoint_sha256\": {\"heads.pt\": \"8994fda7fdb22667eece5a84766238e08d3710a2fa23e9bb9c43c75fd876d205\", \"adapter/adapter_model.safetensors\": \"63fd0bbdf7dcdea5b938ef7536016db23e1edc8ed1f15c5f1d21b682ffd1736e\", \"adapter/README.md\": \"43064eed5a371c72e6afd650a4d8f98837b6026ba9068fbf62ef1887fb70cf55\", \"adapter/adapter_config.json\": \"abb8632c980385f5503d1c4acbd9cf4bd4838277967a0d9115d178922d860b61\"}, \"hidden_size\": 4096, \"origin_version\": 25, \"origin_worker_sha256\": \"12c5b7a5f7fb1384dbbd864134b281fe8d3699b61682a598c79aa2b62b722f6d\", \"origin_readout_sha256\": \"9f6ba4e1bbf6cd0556a53b625819f280609044f8e84579da36dcf31bda95dc66\"}, \"4b_small_supcon\": {\"parameters\": {\"experiment\": \"improvement-v18\", \"kind\": \"paired training-objective pilot\", \"model\": \"qwen-lm/qwen-3/Transformers/4b-base/1\", \"seed\": 1801, \"pilot_fold\": 0, \"external_cap_per_class\": 250, \"external_host_cosine_exclusion\": 0.3, \"max_length\": 256, \"batch_size\": 4, \"gradient_accumulation\": 8, \"lora_rank\": 16, \"lora_alpha\": 32, \"target_modules\": [\"q_proj\", \"v_proj\"], \"pool_epochs\": 1, \"host_epochs\": 2, \"pool_adapter_lr\": 0.0001, \"pool_head_lr\": 0.0005, \"host_adapter_lr\": 3e-05, \"host_head_lr\": 0.0001, \"contrastive_weight\": 0.1, \"contrastive_temperature\": 0.1, \"variants\": [\"ce\", \"ce_supcon\"], \"checkpoint_rule\": \"Evaluate fixed final host epoch only; no best-epoch selection\", \"promotion\": \"SupCon must improve final pilot accuracy by at least 0.01 over identical CE training before broader confirmation. Neither pilot alone qualifies for submission.\", \"wall_limit_seconds\": 24300, \"sources\": [\"https://arxiv.org/html/2004.11362v5\", \"https://github.com/HobbitLong/SupContrast\", \"https://arxiv.org/html/2106.09685v2\", \"https://github.com/microsoft/LoRA\", \"https://huggingface.co/docs/peft/package_reference/lora\"], \"attribution\": \"LoRA and supervised contrastive learning are established methods. Joint CE plus cross-prompt supervised contrastive adaptation here is our experimental combination, not a reproduction of the image-learning paper. Public pool credited to Nishant Dahal and original contributors (CC BY-SA 4.0).\", \"limitations\": [\"One repeatedly inspected host fold, not untouched confirmation\", \"4500-row external cap screens objectives before scaling\", \"Small contrastive batches may limit the benefit\", \"Host-based external near-match exclusion uses unlabeled host text only for conservative leakage screening\"], \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"pool_sha256\": \"37750e899ff6634c147daa7888bfed048e92d3fb1381fe5d21f93bf72a84e8ec\", \"folds_sha256\": \"c64582951ffc6bd6a8233f97775fede7404a52ee5b73c88fcf18b8271fc80508\", \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"]}, \"variant\": \"ce_supcon\", \"readout_variant\": \"supcon_final\", \"training_worker\": \"4b_small_supcon_worker.py\", \"readout_worker\": \"4b_small_supcon_readout.py\", \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8}, \"checkpoint_sha256\": {\"heads.pt\": \"6c540f09084be67029f3651a020214fcb2afe53e95a127486985bf521a7e10ff\", \"adapter/adapter_model.safetensors\": \"353f3e3bacab0660d60b8b8e4a90e4d258f26fa5047a4010db3f2227b7bfb6d5\", \"adapter/README.md\": \"c7094bd33e036ce53b5d373d4c251d03e96d7b624f3ce89c9805e5f5844dd79b\", \"adapter/adapter_config.json\": \"5cc84f15952fcb3d9c192cc8d819fc201a31c73f95884656c5fe2acabda33ab7\"}, \"hidden_size\": 2560, \"origin_version\": 18, \"origin_worker_sha256\": \"a79591118969d7e2b687523a64c46f68399a217c36972d37e300aaba7d440b34\", \"origin_readout_sha256\": \"c55bed7580eefde8f7461fff084510cd2c83fd5e0768cff066ae1ff700a4409c\"}}, \"source_sha256\": {\"run_confirmation_arm_v34.py\": \"91a03e111eda97d4e7c9e0203c87241487ceb39c7d7e18027c80cb0793a3fe91\", \"prepare_likelihood_v33.py\": \"9517acff96c5f73c2838a3cc06bdc20aadeff9e14c801b460e3d6cd28c222838\", \"8b_ce_worker.py\": \"33c117d136081b7a330becdcd8232bef58eae7b5611c4d807fffb6cf9ea65c82\", \"8b_ce_readout.py\": \"fa1fe90c97657ac6b06175192244dfb53ecb7dc18f946975cb722f1caaa33662\", \"4b_small_supcon_worker.py\": \"6d3371ddfef1c49d2a364c5e0b0f9152095c39d3c5302e2f8bdeb262ab16d96c\", \"4b_small_supcon_readout.py\": \"60ca39dfd35b1be99d73c0671776eee57feb193b480c20ba101c7d7761770fbb\"}, \"primary_audit_sha256\": \"c08145212c44ab6b3b6e0064db5178ece7a01ec6005477ec97c356f467352ead\", \"primary_oof_sha256\": \"6609dcecf4cd18bc5fbb901794feb90a7e2e9f78485f572942820b30258c4021\", \"confirmation_kind\": \"related\", \"confirmation_folds\": [0]}")

import torch
assert torch.cuda.device_count()==2
hosts=list(Path('/kaggle/input').rglob('train.csv'))
assert len(hosts)==1
assert hashlib.sha256(hosts[0].read_bytes()).hexdigest()==plan['host_sha256']
plan['host_path']=str(hosts[0])
for arm,spec in plan['arms'].items():
    candidates=[]
    for p in Path('/kaggle/input').rglob('config.json'):
        cfg=json.loads(p.read_text())
        if cfg.get('model_type')=='qwen3' and cfg.get('hidden_size')==spec['hidden_size'] and cfg.get('num_hidden_layers')==36:
            candidates.append(p.parent)
    assert len(candidates)==1,(arm,candidates)
    spec['model_dir']=str(candidates[0])
    weights=[p for p in Path('/kaggle/input').rglob('adapter_model.safetensors')
             if p.parent.parent.name=='external_only' and p.parent.parent.parent.name==spec['variant']
             and hashlib.sha256(p.read_bytes()).hexdigest()==spec['checkpoint_sha256']['adapter/adapter_model.safetensors']]
    assert len(weights)==1,(arm,len(weights))
    spec['checkpoint']=str(weights[0].parent.parent)
    for relative,digest in spec['checkpoint_sha256'].items():
        assert hashlib.sha256((Path(spec['checkpoint'])/relative).read_bytes()).hexdigest()==digest
plan['deadline_unix']=time.time()+plan['protocol']['runtime']['wall_limit_seconds']
Path('plan.json').write_text(json.dumps(plan,indent=2))
processes=[]
try:
    for gpu,arm in enumerate(plan['arms']):
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu))
        processes.append(subprocess.Popen([sys.executable,'-u','run_confirmation_arm_v34.py',arm],env=env,start_new_session=True))
    while any(p.poll() is None for p in processes):
        if any(p.poll() not in (None,0) for p in processes):raise RuntimeError('Confirmation arm failed')
        if time.time()>plan['deadline_unix']:raise TimeoutError('Confirmation notebook wall budget exhausted')
        time.sleep(10)
    assert all(p.returncode==0 for p in processes)
except BaseException:
    for p in processes:
        if p.poll() is None:os.killpg(p.pid,signal.SIGTERM)
    for p in processes:p.wait()
    raise
summary={'completed':True,'experiment':'improvement-v34','submission_authorized':False,
         'decision':'Independent related-holdout audit and frozen related gates required'}
Path('summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary),flush=True)
